# Practice Assignment - LLM Inference with vLLM

### Experiments
1. Observe prefill and decode through TTFT and generation time.
2. Explore the throughput and latency trade-off when requests arrive concurrently.
3. Measure what automatic prefix caching saves.

The notebook includes the server, streaming, and concurrency plumbing. The experiment code is left incomplete and must be finished by you.


## Before you start

Use **Runtime > Change runtime type > GPU**.

For every experiment:

- Predict before running the measurement.
- Change only the variable named in the task.
- Use repeated measurements rather than one request.
- Compare results from your own Colab runtime. Different students may receive different GPUs.

If a TODO is difficult, open Hint 1 first. Use Hint 2 only if you still need help.


## Setup

The next cells are **plumbing code**. Run them as-is. You are not expected to write server startup code, HTTP streaming code, or concurrency code.


In [1]:
# Run this before importing torch or vllm.
# Run this before importing torch or vllm.
!pip -q install -U uv
!uv pip install --system -U \
    "https://github.com/vllm-project/vllm/releases/download/v0.23.0/vllm-0.23.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl" \
    "numpy==2.0.2" \
    --torch-backend=cu129 \
    --extra-index-url https://download.pytorch.org/whl/cu129 \
    --index-strategy unsafe-best-match



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 41.6 MB/s eta 0:00:00
Using Python 3.12.13 environment at: /usr
Resolved 194 packages in 26.91s
Prepared 133 packages in 2m 06s
Uninstalled 64 packages in 1.42s
Installed 133 packages in 1.74s
 - aiohttp==3.14.1
 + aiohttp==3.14.3
 - annotated-doc==0.0.4
 + annotated-doc==0.0.5
 - annotated-types==0.7.0
 + annotated-types==0.8.0
 + anthropic==0.121.0
 + apache-tvm-ffi==0.1.9
 + astor==0.8.1
 + blake3==1.0.9
 - cachetools==6.2.6
 + cachetools==7.1.7
 + cbor2==6.1.4
 - certifi==2026.6.17
 + certifi==2026.7.22
 - cffi==2.1.0
 + cffi==2.1.1
 - charset-normalizer==3.4.9
 + charset-normalizer==3.5.0
 + compressed-tensors==0.17.0
 - cryptography==49.0.0
 + cryptography==50.0.0
 - cuda-pathfinder==1.5.6
 + cuda-pathfinder==1.6.0
 + cuda-tile==1.3.0
 - cuda-toolkit==12.8.1
 + cuda-toolkit==12.9.1
 + depyf==0.20.0
 + detect-installer==0.1.0
 - dill==0.3.8
 + dill==0.4.1
 + diskcache==5.6.3
 + dnspython==2.8.0
 + email-validator==2.3.0
 - f

In [2]:
import os
import gc
import json
import math
import statistics
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import requests
import torch
from transformers import AutoTokenizer

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime and run the notebook again")

import vllm

print("Python", sys.version.split()[0])
print("PyTorch", torch.__version__)
print("vLLM", vllm.__version__)
print("GPU", torch.cuda.get_device_name(0))
print("CUDA", torch.version.cuda)

Python 3.12.13
PyTorch 2.11.0+cu129
vLLM 0.23.0
GPU Tesla T4
CUDA 12.9


In [3]:
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
PORT = 8000
MAX_MODEL_LEN = 4096
GPU_MEMORY_UTILIZATION = 0.78

RESULTS_DIR = Path("/content/vllm_lab_results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

### Provided server and measurement utilities

`stream_completion` returns a dictionary with the following metrics:

- `ttft_ms`: client-observed time to first streamed text token (milliseconds)
- `e2e_ms`: total request time (milliseconds)
- `approx_tpot_ms`: approximate time per output token after the first token (milliseconds)
- `output_tokens`: approximate generated token count

`run_concurrent_requests` handles the threading needed to submit requests concurrently. You will design the experiment around it.


In [4]:
SERVER_PROC = None
SERVER_LOG_HANDLE = None


def _kill_stale_vllm():
    """Remove an orphaned vLLM server from an earlier notebook run."""
    # First try to kill whatever owns the notebook's server port.
    try:
        subprocess.run(
            ["fuser", "-k", f"{PORT}/tcp"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            check=False,
        )
    except FileNotFoundError:
        pass

    # Fallback for Colab environments where fuser is unavailable.
    try:
        subprocess.run(
            ["pkill", "-f", rf"vllm.*serve.*--port.*{PORT}"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            check=False,
        )
    except FileNotFoundError:
        pass

    time.sleep(2)


def stop_server():
    global SERVER_PROC, SERVER_LOG_HANDLE

    if SERVER_PROC is not None and SERVER_PROC.poll() is None:
        SERVER_PROC.terminate()
        try:
            SERVER_PROC.wait(timeout=20)
        except subprocess.TimeoutExpired:
            SERVER_PROC.kill()
            SERVER_PROC.wait(timeout=10)

    SERVER_PROC = None

    if SERVER_LOG_HANDLE is not None:
        SERVER_LOG_HANDLE.close()
        SERVER_LOG_HANDLE = None

    # Important in notebooks: rerunning the setup cell can lose SERVER_PROC
    # while the old vLLM process is still alive.
    _kill_stale_vllm()

    gc.collect()
    torch.cuda.empty_cache()
    time.sleep(2)


def show_server_log_tail(lines=50):
    log_path = RESULTS_DIR / "vllm_server.log"

    if SERVER_LOG_HANDLE is not None:
        try:
            SERVER_LOG_HANDLE.flush()
        except Exception:
            pass

    if log_path.exists():
        text = log_path.read_text(errors="replace")
        print("\n".join(text.splitlines()[-lines:]))


def start_server(prefix_caching=False, timeout_s=600):
    global SERVER_PROC, SERVER_LOG_HANDLE

    stop_server()

    log_path = RESULTS_DIR / "vllm_server.log"
    SERVER_LOG_HANDLE = open(log_path, "w", buffering=1)

    cmd = [
        sys.executable,
        "-m",
        "vllm.entrypoints.cli.main",
        "serve",
        MODEL,
        "--host",
        "127.0.0.1",
        "--port",
        str(PORT),
        "--dtype",
        "float16",
        "--gpu-memory-utilization",
        str(GPU_MEMORY_UTILIZATION),
        "--max-model-len",
        str(MAX_MODEL_LEN),
    ]

    if prefix_caching:
        cmd.append("--enable-prefix-caching")
    else:
        cmd.append("--no-enable-prefix-caching")

    print("=" * 80)
    print("Starting vLLM server")
    print("=" * 80)
    print("Model:", MODEL)
    print("Port:", PORT)
    print("Prefix caching:", prefix_caching)
    print("GPU memory utilization:", GPU_MEMORY_UTILIZATION)
    print("Max model length:", MAX_MODEL_LEN)
    print()
    print("Command:")
    print(" ".join(cmd))
    print()
    print("-" * 80)
    print("vLLM LOG")
    print("-" * 80)

    # Capture stdout so we can both display it live and save it.
    SERVER_PROC = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env={
            **os.environ,
            "PYTHONUNBUFFERED": "1",

            # Avoid FlashInfer JIT compilation during startup.
            "VLLM_USE_FLASHINFER_SAMPLER": "0",
        },
    )

    # Read vLLM's stdout continuously in a separate thread.
    import threading

    def stream_logs():
        try:
            for line in iter(SERVER_PROC.stdout.readline, ""):
                if not line:
                    break

                # Show it immediately in the notebook.
                print(line, end="", flush=True)

                # Also preserve it in the log file.
                try:
                    SERVER_LOG_HANDLE.write(line)
                    SERVER_LOG_HANDLE.flush()
                except (ValueError, OSError):
                    pass
        finally:
            try:
                SERVER_PROC.stdout.close()
            except Exception:
                pass

    log_thread = threading.Thread(
        target=stream_logs,
        daemon=True,
    )
    log_thread.start()

    health_url = f"http://127.0.0.1:{PORT}/v1/models"
    deadline = time.time() + timeout_s

    while time.time() < deadline:
        return_code = SERVER_PROC.poll()

        if return_code is not None:
            # Give the logging thread a moment to print the final lines.
            log_thread.join(timeout=2)

            print()
            print("-" * 80)
            print(f"vLLM exited with code {return_code}")
            print(f"Full log: {log_path}")
            print("-" * 80)

            raise RuntimeError(
                f"vLLM exited early with code {return_code}. "
                f"See {log_path}"
            )

        try:
            response = requests.get(
                health_url,
                timeout=2,
            )

            if response.ok:
                print()
                print("=" * 80)
                print("vLLM SERVER READY")
                print("=" * 80)
                print(f"Endpoint: http://127.0.0.1:{PORT}")
                print(f"Logs: {log_path}")
                print()

                return

        except requests.RequestException:
            # Normal while vLLM is still initializing.
            pass

        time.sleep(2)

    print()
    print("=" * 80)
    print(f"STARTUP TIMEOUT after {timeout_s} seconds")
    print("=" * 80)

    stop_server()

    raise TimeoutError(
        f"vLLM did not become ready within {timeout_s} seconds. "
        f"See {log_path}"
    )

In [5]:
def stream_completion(prompt, max_tokens=16):
    url = f"http://127.0.0.1:{PORT}/v1/completions"
    payload = {
        "model": MODEL,
        "prompt": prompt,
        "max_tokens": max_tokens,
        "temperature": 0.0,
        "ignore_eos": True,
        "stream": True,
    }

    start = time.perf_counter()
    first_text_time = None
    pieces = []

    with requests.post(url, json=payload, stream=True, timeout=180) as response:
        response.raise_for_status()
        for raw in response.iter_lines():
            if not raw or not raw.startswith(b"data: "):
                continue
            data = raw[len(b"data: ") :]
            if data == b"[DONE]":
                break
            obj = json.loads(data)
            text = obj["choices"][0].get("text", "")
            if text:
                if first_text_time is None:
                    first_text_time = time.perf_counter()
                pieces.append(text)

    end = time.perf_counter()
    generated_text = "".join(pieces)
    output_tokens = len(tokenizer.encode(generated_text, add_special_tokens=False))
    ttft_s = first_text_time - start if first_text_time is not None else float("nan")
    e2e_s = end - start
    tpot_s = (e2e_s - ttft_s) / max(output_tokens - 1, 1)

    return {
        "ttft_ms": 1000 * ttft_s,
        "e2e_ms": 1000 * e2e_s,
        "approx_tpot_ms": 1000 * tpot_s,
        "output_tokens": output_tokens,
        "text": generated_text,
    }


def repeat_request(prompt, max_tokens, repeats=3):
    values = [stream_completion(prompt, max_tokens=max_tokens) for _ in range(repeats)]
    return {
        "median_ttft_ms": statistics.median(v["ttft_ms"] for v in values),
        "median_e2e_ms": statistics.median(v["e2e_ms"] for v in values),
        "median_tpot_ms": statistics.median(v["approx_tpot_ms"] for v in values),
        "median_output_tokens": statistics.median(v["output_tokens"] for v in values),
    }


def run_concurrent_requests(prompts, concurrency, max_tokens=32):
    start = time.perf_counter()
    results = []
    with ThreadPoolExecutor(max_workers=concurrency) as pool:
        futures = [pool.submit(stream_completion, p, max_tokens) for p in prompts]
        for future in as_completed(futures):
            results.append(future.result())
    wall_s = time.perf_counter() - start
    return results, wall_s

### Start the shared server

We begin with prefix caching disabled. Experiments 1 and 2 use this same server so setup time does not dominate the lab.


In [6]:
start_server(prefix_caching=False)
_ = stream_completion("Warm up the inference server", max_tokens=8)
print("READY")

Starting vLLM server
Model: Qwen/Qwen2.5-0.5B-Instruct
Port: 8000
Prefix caching: False
GPU memory utilization: 0.78
Max model length: 4096

Command:
/usr/bin/python3 -m vllm.entrypoints.cli.main serve Qwen/Qwen2.5-0.5B-Instruct --host 127.0.0.1 --port 8000 --dtype float16 --gpu-memory-utilization 0.78 --max-model-len 4096 --no-enable-prefix-caching

--------------------------------------------------------------------------------
vLLM LOG
--------------------------------------------------------------------------------
(APIServer pid=3411) INFO 08-13 05:16:14 [api_utils.py:339] 
(APIServer pid=3411) INFO 08-13 05:16:14 [api_utils.py:339]        █     █     █▄   ▄█
(APIServer pid=3411) INFO 08-13 05:16:14 [api_utils.py:339]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.23.0
(APIServer pid=3411) INFO 08-13 05:16:14 [api_utils.py:339]   █▄█▀ █     █     █     █  model   Qwen/Qwen2.5-0.5B-Instruct
(APIServer pid=3411) INFO 08-13 05:16:14 [api_utils.py:339]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=

# Experiment 1 - Can you observe prefill and decode?

**Concept recap:** Prefill processes the input prompt and produces the first output token. Decode then generates later tokens sequentially. Time to first token (TTFT) is therefore strongly connected to prefill, while time per output token is connected to decode.

You will run two controlled experiments:

- Change prompt length while keeping output length short.
- Change output length while keeping the prompt fixed.

### Your Prediction
Before coding, which measurement should react most clearly to a much longer prompt? Which measurement should react most clearly to a much longer generated output?


> **Your response**
>
> Write your response here


### Task 1A - Vary prompt length

Complete the cell. Create four prompts of increasing length, request exactly 8 output tokens, and store the prompt token count and median measurements in `prefill_rows`.


In [ ]:
prompt_repeats = [4, 16, 48, 96]
prefill_rows = []

for repeats in prompt_repeats:
    # TODO 1: Create a prompt by repeating the sentence below `repeats` times.
    # Add a short final instruction so the request still looks natural.
    base_sentence = "Efficient LLM inference has a prefill phase and a decode phase. "
    prompt = None

    # TODO 2: Use repeat_request with max_tokens=8.
    measurement = None

    # TODO 3: Append a dictionary with these fields:
    # prompt_tokens, median_ttft_ms, median_e2e_ms, median_tpot_ms
    # Replace the pass statement with your code.
    pass

prefill_df = pd.DataFrame(prefill_rows)
display(prefill_df)

<details>
<summary>Hint 1</summary>

Only prompt length should change. The output length must stay at 8 for every row. Use `tokenizer.encode(..., add_special_tokens=False)` to count prompt tokens.

</details>


<details>
<summary>Hint 2</summary>

Pseudocode:

    for each repeat count
        make prompt
        measurement = repeat_request(prompt, 8)
        append prompt token count and measurement values

</details>


In [ ]:
required = {"prompt_tokens", "median_ttft_ms", "median_e2e_ms", "median_tpot_ms"}
if (
    isinstance(prefill_df, pd.DataFrame)
    and len(prefill_df) == 4
    and required.issubset(prefill_df.columns)
):
    if prefill_df["prompt_tokens"].is_monotonic_increasing:
        print("PASS")
    else:
        print("CHECK AGAIN")
else:
    print("CHECK AGAIN")

### Task 1B - Vary output length

Now keep the prompt fixed and vary `max_tokens` across 4, 16, 32, and 64. Store your results in `decode_rows` and create `decode_df`.


In [ ]:
fixed_prompt = (
    "A language model first processes the prompt and then generates tokens one at a time. "
    "Explain why these phases have different performance characteristics."
)
output_lengths = [4, 16, 32, 64]
decode_rows = []

# TODO: Write the loop. Store max_tokens, median_ttft_ms,
# median_e2e_ms, and median_tpot_ms for each output length.


decode_df = pd.DataFrame(decode_rows)
display(decode_df)

<details>
<summary>Hint 1</summary>

The prompt must be identical on every iteration. The only independent variable is `max_tokens`.

</details>


<details>
<summary>Hint 2</summary>

Pseudocode:

    for each output length
        measurement = repeat_request(fixed_prompt, output length)
        append output length and measurement values

</details>


In [ ]:
required = {"max_tokens", "median_ttft_ms", "median_e2e_ms", "median_tpot_ms"}
if (
    isinstance(decode_df, pd.DataFrame)
    and len(decode_df) == 4
    and required.issubset(decode_df.columns)
):
    if sorted(decode_df["max_tokens"].tolist()) == [4, 16, 32, 64]:
        print("PASS")
    else:
        print("CHECK AGAIN")
else:
    print("CHECK AGAIN")

### Explain Experiment 1

1. What happened to TTFT as prompt length increased?
2. What happened to end-to-end latency as output length increased?
3. Why can one end-to-end timer hide whether time is being spent in prefill or decode?
4. Did your measurements perfectly match your prediction? If not, give one plausible source of noise.


> **Your response**
>
> Use your measured values in the explanation.


# Experiment 2 - Does more concurrency make inference faster?

**Concept recap:** Inference engines batch and schedule work from multiple requests to use the GPU efficiently. Higher concurrency can increase aggregate throughput, but an individual request may wait longer or compete for resources. Throughput and latency therefore answer different questions.

### Prediction
As concurrency increases from 1 to 8, what do you expect to happen to output tokens per second and median TTFT?


> **Your response**
>
> Describe the expected direction of both metrics.


### Task 2 - Design the concurrency experiment

The helper `run_concurrent_requests(prompts, concurrency, max_tokens)` is provided. It returns the completed request dictionaries and total wall time.

For each concurrency level 1, 2, 4, and 8:

- create enough unique prompts to keep workers busy;
- call the helper;
- calculate aggregate output tokens per second;
- calculate median TTFT and median end-to-end latency;
- append one row to `concurrency_rows`.


In [ ]:
concurrency_levels = [1, 2, 4, 8]
concurrency_rows = []

for concurrency in concurrency_levels:
    total_requests = max(8, 2 * concurrency)
    prompts = [
        f"Unique request {i} at concurrency {concurrency}. Explain one inference metric briefly."
        for i in range(total_requests)
    ]

    # TODO 1: Call run_concurrent_requests with max_tokens=32.
    results = None
    wall_s = None

    # TODO 2: Calculate total output tokens / wall_s.
    output_tokens_per_s = None

    # TODO 3: Calculate median TTFT and median E2E from `results`.
    median_ttft_ms = None
    median_e2e_ms = None

    # TODO 4: Append one dictionary to concurrency_rows.

concurrency_df = pd.DataFrame(concurrency_rows)
display(concurrency_df)

<details>
<summary>Hint 1</summary>

Each element of `results` has the keys `output_tokens`, `ttft_ms`, and `e2e_ms`. Use `sum` for total tokens and `statistics.median` for latency.

</details>


<details>
<summary>Hint 2</summary>

Pseudocode:

    results, wall = run concurrent requests
    throughput = sum output tokens divided by wall
    median ttft = median of request ttft values
    median e2e = median of request e2e values
    append one summary row

</details>


In [ ]:
required = {"concurrency", "output_tokens_per_s", "median_ttft_ms", "median_e2e_ms"}
if (
    isinstance(concurrency_df, pd.DataFrame)
    and len(concurrency_df) == 4
    and required.issubset(concurrency_df.columns)
):
    if sorted(concurrency_df["concurrency"].tolist()) == [1, 2, 4, 8]:
        print("PASS")
    else:
        print("CHECK AGAIN")
else:
    print("CHECK AGAIN")

### Visualize and explain

Create one plot for throughput versus concurrency and one plot for median TTFT versus concurrency. Choose clear axis labels.


In [ ]:
# TODO: Create two plots from concurrency_df.
# You may use pandas plotting or matplotlib directly.

### Explain Experiment 2

1. Did throughput increase linearly with concurrency?
2. What happened to median TTFT?
3. Is the concurrency setting with the highest throughput automatically the best setting for an interactive application? Why?


> **Your response**
>
> Use both system throughput and user-facing latency in your answer.


# Experiment 3 - What does prefix caching save?

**Concept recap:** If later requests begin with the same long prefix, vLLM can reuse cached KV state for that prefix instead of repeating all of its prefill computation. This should mainly affect the work done before the first new token is produced.

### Prediction
Which metric should show the clearest improvement when a long prefix is reused: TTFT, TPOT, or both equally? Explain why before running the comparison.


> **Your response**
>
> Write your response here.


### Task 3A - Build a cacheable workload

Create one long `shared_prefix` and at least four different suffix questions. The combined prompts must all begin with exactly the same `shared_prefix`.


In [ ]:
base_prefix_text = (
    "Course note. Prefill processes prompt tokens in parallel. Decode generates later tokens "
    "sequentially. KV caching stores key and value representations from previous tokens. "
    "Prefix caching can reuse KV state across requests that begin with the same token prefix. "
)

# TODO 1: Build a long shared prefix by repeating base_prefix_text.
# Aim for roughly 800 to 1600 tokens so the effect is visible but fits MAX_MODEL_LEN.
shared_prefix = None

suffixes = [
    "Define TTFT in one sentence.",
    "Which phase is most related to TTFT?",
    "What information is stored in a KV cache?",
    "Why can a shared prefix reduce repeated work?",
]

# TODO 2: Build `prefix_prompts` by combining the identical prefix with each suffix.
prefix_prompts = []

if shared_prefix is not None:
    print(
        "Shared prefix tokens",
        len(tokenizer.encode(shared_prefix, add_special_tokens=False)),
    )

<details>
<summary>Hint 1</summary>

The prefix must be byte-for-byte identical before every suffix. Repeating the provided paragraph many times is enough for this lab.

</details>


<details>
<summary>Hint 2</summary>

Pseudocode:

    shared prefix = base text repeated N times
    prompts = shared prefix plus each suffix

</details>


In [ ]:
if isinstance(shared_prefix, str) and len(prefix_prompts) >= 4:
    token_count = len(tokenizer.encode(shared_prefix, add_special_tokens=False))
    same_prefix = all(p.startswith(shared_prefix) for p in prefix_prompts)
    within_limit = token_count < MAX_MODEL_LEN - 64
    if token_count >= 800 and same_prefix and within_limit:
        print("PASS")
    else:
        print("CHECK AGAIN")
else:
    print("CHECK AGAIN")

### Task 3B - Measure prefix caching OFF

The current server still has prefix caching disabled. Measure the four prompts with a short output of 8 tokens and store the request dictionaries in `prefix_off_results`.


In [ ]:
# TODO: Send each prompt once with max_tokens=8 and store the results.
prefix_off_results = []

# Optional: display a small table after you have filled the list.

In [ ]:
if (
    isinstance(prefix_off_results, list)
    and len(prefix_off_results) == len(prefix_prompts)
    and len(prefix_off_results) >= 4
):
    if all(isinstance(r, dict) and "ttft_ms" in r for r in prefix_off_results):
        print("PASS")
    else:
        print("CHECK AGAIN")
else:
    print("CHECK AGAIN")

### Switch the server to prefix caching ON

This is provided plumbing code. Run it as-is. After the restart, **you** must prime the cache before measuring.


In [ ]:
start_server(prefix_caching=True)
_ = stream_completion("Unrelated server warm up", max_tokens=8)
print("READY")

### Task 3C - Prime and measure prefix caching ON

First send one request that contains the shared prefix so vLLM has seen that prefix. Do not include this priming request in your measurements. Then measure the same `prefix_prompts` with the same `max_tokens=8` and store the results in `prefix_on_results`.


In [ ]:
# TODO 1: Send one priming request containing shared_prefix.

# TODO 2: Measure the same prefix_prompts with max_tokens=8.
prefix_on_results = []

<details>
<summary>Hint 1</summary>

The priming request must begin with `shared_prefix`. Its suffix can be something simple such as a warm-up instruction. The measured prompts must be exactly the same ones used in the OFF condition.

</details>


<details>
<summary>Hint 2</summary>

Pseudocode:

    send shared prefix plus one warm up suffix
    for each measured prompt
        send prompt with 8 output tokens
        store returned dictionary

</details>


In [ ]:
if (
    isinstance(prefix_on_results, list)
    and len(prefix_on_results) == len(prefix_prompts)
    and len(prefix_on_results) >= 4
):
    if all(isinstance(r, dict) and "ttft_ms" in r for r in prefix_on_results):
        print("PASS")
    else:
        print("CHECK AGAIN")
else:
    print("CHECK AGAIN")

### Task 3D - Compare the conditions

Create a two-row dataframe called `prefix_summary` with these columns:

- `condition`
- `median_ttft_ms`
- `median_e2e_ms`
- `median_tpot_ms`

Use medians across the measured requests.


In [ ]:
# TODO: Summarize prefix_off_results and prefix_on_results.
prefix_summary = pd.DataFrame()

display(prefix_summary)

In [ ]:
required = {"condition", "median_ttft_ms", "median_e2e_ms", "median_tpot_ms"}
if (
    isinstance(prefix_summary, pd.DataFrame)
    and len(prefix_summary) == 2
    and required.issubset(prefix_summary.columns)
):
    print("PASS")
else:
    print("CHECK AGAIN")

### Explain Experiment 3

1. Which metric changed most clearly when prefix caching was enabled?
2. Why did we deliberately keep the generated output short?
3. Why would it be a bad experiment to change GPU memory utilization at the same time as the prefix caching flag?
4. If your TTFT improvement is small or noisy, what could you change about the workload without changing the concept being tested?


> **Your response**
>
> Explain the mechanism.


# vLLM synthesis

Complete the table in your own words.

| What you changed | Main work affected | Metric that should react | What your data showed |
|---|---|---|---|
| Longer prompt | | | |
| Longer output | | | |
| Higher concurrency | | | |
| Prefix caching ON | | | |

### Before moving to the torch.compile notebook

**Why is measuring only total response time not enough to understand an inference optimization?**


> **Your response**
>
>


## Extra practice if you finish early

Design one additional experiment that changes **only one variable**. Examples include a longer shared prefix, a higher concurrency level, or a different output length. State the hypothesis before running it and identify the metric that should react most strongly.


In [ ]:
# Optional practice experiment workspace

## Cleanup before the torch.compile notebook

Run this cell before moving on. It stops the vLLM server and releases its GPU memory.


In [ ]:
stop_server()
print("CLEANED UP")